# Module C (Can/Square): JEPA-Augmented BC

**Behavior Cloning with JEPA Data Augmentation for Can and Square tasks.**

Replicates C1-C3 from `module_c_bc_repa.ipynb` (Lift) for the Can and Square Robomimic tasks.
Select task below, then run all cells sequentially.

| # | Task | Detail |
|---|------|--------|
| C1 | BC policy baseline | Train MLP head (DINOv2 embedding → 7D action) on real data |
| C2 | Synthetic trajectory generation | Perturb embeddings, roll out predictor with expert actions, store $(z', a)$ pairs |
| C3 | JEPA-augmented BC training | Train BC on real + synthetic at $\alpha \in \{0.0,0.25,0.5,0.75,1.0\}$, select best $\sigma$ |

**Prerequisites**:
- Can/Square embeddings cached at `data/embeddings/embeddings_{TASK}_{train,val}.pt` (from `Can_Square_Hier_experiment.ipynb` S3)
- Can/Square triplets at `data/embeddings/triplets_{TASK}_*_K{K}.pt` (from S4)
- Can/Square predictor checkpoint at `outputs/transformer_{TASK}_K1_T8.pt` (from S5, used for C2 synthetic rollout)
- Can/Square norm stats at `data/embeddings/norm_stats_{TASK}.pt` (from S3)


## Section 1: Setup & Imports

In [ ]:
# ── TASK SELECTOR ──
TASK = 'can'  # 'can' or 'square'
print(f'Selected task: {TASK.upper()}')

In [ ]:
import sys, subprocess

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['torch>=2.0', 'matplotlib', 'scikit-learn', 'tqdm', 'numpy', 'pandas', 'scipy', 'pyarrow']:
    pip_install(pkg)

import os, math, random, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Paths
ROOT = Path(os.getcwd()).resolve()
if not (ROOT / 'data' / 'embeddings').exists():
    if (ROOT.parent / 'data' / 'embeddings').exists():
        ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
OUTPUT_DIR = ROOT / 'outputs'
EMBED_DIR = DATA_DIR / 'embeddings'
DATASET_DIR = DATA_DIR / f'robomimic-ph-{TASK}-image'
PARQUET_DIR = DATASET_DIR / 'data' / 'chunk-000'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Config (mirrors module_c_bc_repa.ipynb, adapted for Can/Square)
class Config:
    embed_dim = 384
    action_dim = 7
    d_model = 256
    n_heads = 4
    n_layers_tr = 4
    batch_size = 64
    lr = 1e-4
    weight_decay = 0.05
    warmup_steps = 200
    max_epochs = 50
    use_amp = True
    dropout_p = 0.2
    seed = 42
    K_values = [1, 5, 10, 20]
    best_T = 8
    n_train_episodes = 160
    # BC
    bc_hidden = [256, 128]
    bc_epochs = 30
    bc_batch_size = 64
    # Synthetic generation
    N_perturb = 10
    sigma_values = [0.01, 0.05, 0.1]
    rollout_K = 1
    rollout_T = 8
    # JEPA-augmented BC
    alpha_values = [0.0, 0.25, 0.5, 0.75, 1.0]

cfg = Config()

# Device & reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(cfg.seed)
    torch.backends.cudnn.benchmark = True

print('Setup complete.')

## Section 2: Load Cached Data

Load pre-computed Can/Square embeddings, actions, norm stats, and per-episode Parquet data.
These are produced by `Can_Square_Hier_experiment.ipynb` S3-S6.

In [ ]:
# Task-specific paths
EMB_TRAIN_PATH = EMBED_DIR / f'embeddings_{TASK}_train.pt'
EMB_VAL_PATH = EMBED_DIR / f'embeddings_{TASK}_val.pt'
ACT_TRAIN_PATH = EMBED_DIR / f'actions_{TASK}_train.pt'
ACT_VAL_PATH = EMBED_DIR / f'actions_{TASK}_val.pt'
NORM_STATS_PATH = EMBED_DIR / f'norm_stats_{TASK}.pt'

assert EMB_TRAIN_PATH.exists(), (
    f'Missing {EMB_TRAIN_PATH}. Run Can_Square_Hier_experiment.ipynb S1-S3 for TASK={TASK!r} first.'
)
assert EMB_VAL_PATH.exists(), f'Missing {EMB_VAL_PATH}'

train_emb = torch.load(EMB_TRAIN_PATH).float()
val_emb = torch.load(EMB_VAL_PATH).float()
train_act = torch.load(ACT_TRAIN_PATH).float()
val_act = torch.load(ACT_VAL_PATH).float()
norm_stats = torch.load(NORM_STATS_PATH, weights_only=False)

print(f'Train embeddings: {train_emb.shape}')
print(f'Val embeddings:   {val_emb.shape}')
print(f'Train actions:    {train_act.shape}')

# Load all Parquet files for episode boundaries + raw actions (for synthetic gen)
parquet_files = sorted(PARQUET_DIR.glob('episode_*.parquet'))
print(f'Found {len(parquet_files)} parquet episodes')

all_ep_actions = []
all_ep_n_frames = []

for pf in tqdm(parquet_files, desc='Loading parquets'):
    df = pd.read_parquet(pf)
    actions_raw = np.stack(df['action'].values).astype(np.float32)
    all_ep_actions.append(torch.tensor(actions_raw[::2]))
    all_ep_n_frames.append(len(actions_raw[::2]))

# Verify train/val split
train_frames_total = sum(all_ep_n_frames[:cfg.n_train_episodes])
val_frames_total = sum(all_ep_n_frames[cfg.n_train_episodes:])
assert train_frames_total == train_emb.shape[0], f'Train mismatch: {train_frames_total} vs {train_emb.shape[0]}'
assert val_frames_total == val_emb.shape[0], f'Val mismatch: {val_frames_total} vs {val_emb.shape[0]}'
print(f'Train: {train_frames_total} frames in {cfg.n_train_episodes} episodes')
print(f'Val:   {val_frames_total} frames in {len(parquet_files) - cfg.n_train_episodes} episodes')

# Episode → embedding index mapping
train_offset = 0; val_offset = 0
ep_to_emb_start = {}
for ep_idx in range(len(parquet_files)):
    n_frames = all_ep_n_frames[ep_idx]
    if ep_idx < cfg.n_train_episodes:
        ep_to_emb_start[ep_idx] = ('train', train_offset)
        train_offset += n_frames
    else:
        ep_to_emb_start[ep_idx] = ('val', val_offset)
        val_offset += n_frames

def get_emb_idx(ep_idx, local_frame_idx):
    split_name, start = ep_to_emb_start[ep_idx]
    return split_name, start + local_frame_idx

mean_embed = train_emb.mean(dim=0, keepdim=True)
print('Data loaded and verified.')

## Section 3: C1 — BC Policy Baseline

Train a simple MLP policy that maps a DINOv2 embedding (384D) to a 7D action via MSE loss.
Architecture: $384 \to 256 \to 128 \to 7$ with ReLU activations.

In [ ]:
class BCDataset(Dataset):
    """Dataset for Behavior Cloning: pairs embeddings with corresponding actions."""
    def __init__(self, embeddings, actions, ep_n_frames):
        self.embeddings = embeddings
        self.actions = actions
        self.samples = []
        offset = 0
        for n_frames in ep_n_frames:
            for i in range(n_frames):
                self.samples.append((offset + i, offset + i))
            offset += n_frames

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        emb_idx, act_idx = self.samples[idx]
        return self.embeddings[emb_idx], self.actions[act_idx]


# Build BC datasets
train_ep_frames = all_ep_n_frames[:cfg.n_train_episodes]
val_ep_frames = all_ep_n_frames[cfg.n_train_episodes:]
bc_train_ds = BCDataset(train_emb, train_act, train_ep_frames)
bc_val_ds = BCDataset(val_emb, val_act, val_ep_frames)

bc_train_ldr = DataLoader(bc_train_ds, batch_size=cfg.bc_batch_size, shuffle=True,
                         num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)
bc_val_ldr = DataLoader(bc_val_ds, batch_size=cfg.bc_batch_size, shuffle=False,
                        num_workers=0, pin_memory=(device.type=='cuda'))

print(f'BC train samples: {len(bc_train_ds)}, val samples: {len(bc_val_ds)}')

In [ ]:
class BCPolicy(nn.Module):
    """Simple MLP policy: embedding (384) → action (7)."""
    def __init__(self, embed_dim=384, action_dim=7, hidden_dims=[256, 128]):
        super().__init__()
        layers = []
        in_dim = embed_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(in_dim, h), nn.ReLU()])
            in_dim = h
        layers.append(nn.Linear(in_dim, action_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, z):
        return self.net(z)

bc_model = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                     hidden_dims=cfg.bc_hidden).to(device)
print(f'BC policy params: {sum(p.numel() for p in bc_model.parameters()):,}')

In [ ]:
def bc_train_one_epoch(model, loader, optimizer, scaler, global_step, total_steps):
    model.train()
    epoch_loss = 0.0
    for z, a_target in loader:
        z = z.to(device)
        a_target = a_target.to(device)
        lr = get_lr(global_step, cfg.warmup_steps, total_steps, cfg.lr)
        for pg in optimizer.param_groups:
            pg['lr'] = lr
        if cfg.use_amp and device.type == 'cuda':
            with autocast():
                a_pred = model(z)
                loss = F.mse_loss(a_pred, a_target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            a_pred = model(z)
            loss = F.mse_loss(a_pred, a_target)
            loss.backward()
            optimizer.step()
        optimizer.zero_grad()
        epoch_loss += loss.item()
        global_step += 1
    return epoch_loss / len(loader), global_step


@torch.no_grad()
def bc_compute_val_mse(model, loader):
    model.eval()
    mse_list = []
    for z, a_target in loader:
        z = z.to(device)
        a_target = a_target.to(device)
        a_pred = model(z)
        mse = F.mse_loss(a_pred, a_target).item()
        mse_list.append(mse)
    return np.mean(mse_list)


def bc_train_model(model, train_ldr, val_ldr, epochs=None, verbose=True):
    epochs = epochs or cfg.bc_epochs
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    scaler = GradScaler(enabled=(cfg.use_amp and device.type=='cuda'))
    total_steps = epochs * len(train_ldr)
    gs = 0
    train_losses, val_mses = [], []
    best_mse = float('inf')
    best_state = None
    for epoch in range(1, epochs + 1):
        avg_loss, gs = bc_train_one_epoch(model, train_ldr, optimizer, scaler, gs, total_steps)
        val_mse = bc_compute_val_mse(model, val_ldr)
        train_losses.append(avg_loss)
        val_mses.append(val_mse)
        if val_mse < best_mse:
            best_mse = val_mse
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose and (epoch % max(1, epochs // 10) == 0 or epoch == 1 or epoch == epochs):
            print(f'E {epoch:3d}/{epochs} | loss={avg_loss:.4f} | val_mse={val_mse:.4f} | best={best_mse:.4f}')
    model.load_state_dict(best_state)
    return {'train_losses': train_losses, 'val_mses': val_mses, 'best_mse': best_mse}


# LR scheduler (from module_a)
def cosine_loss(pred, target):
    return 1.0 - F.cosine_similarity(pred, target, dim=-1).mean()

def get_lr(step, warmup_steps, total_steps, base_lr):
    if step < warmup_steps:
        return base_lr * step / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * progress))

print('BC training utilities ready.')

In [ ]:
print(f'Training BC baseline on real data (TASK={TASK.upper()})...')
print('-' * 55)

bc_baseline = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                        hidden_dims=cfg.bc_hidden).to(device)
bc_result = bc_train_model(bc_baseline, bc_train_ldr, bc_val_ldr, epochs=cfg.bc_epochs)

print(f'\nBC baseline best val MSE: {bc_result["best_mse"]:.6f}')

# Save checkpoint (task-specific name)
bc_ckpt = {
    'model_state_dict': bc_baseline.state_dict(),
    'val_mse': bc_result['best_mse'],
    'task': TASK,
    'config': {k: v for k, v in vars(cfg).items() if not k.startswith('__')},
}
baseline_ckpt_path = OUTPUT_DIR / f'module_c_bc_baseline_{TASK}.pt'
torch.save(bc_ckpt, baseline_ckpt_path)
print(f'Saved {baseline_ckpt_path}')

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(bc_result['train_losses'], label='Train MSE')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE')
ax1.set_title(f'C1: BC Training Loss ({TASK.upper()})'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.plot(bc_result['val_mses'], color='orange', label='Val MSE')
ax2.axhline(y=bc_result['best_mse'], color='r', linestyle='--', label=f'Best={bc_result["best_mse"]:.4f}')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('MSE')
ax2.set_title(f'C1: BC Validation MSE ({TASK.upper()})'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'c1_bc_baseline_{TASK}.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 4: C2 — Synthetic Trajectory Generation

For each training episode, perturb the initial embedding $z_0$ with Gaussian noise $\epsilon \sim \mathcal{N}(0, \sigma^2 I)$, then autoregressively roll out the **task-specific** JEPA predictor with expert actions to generate synthetic $(\hat{z}_t, a_t)$ pairs.

Ablate $\sigma \in \{0.01, 0.05, 0.1\}$ with $N_\text{perturb}=10$ per episode.

**Note**: Uses the Can/Square-trained predictor (`transformer_{TASK}_K1_T8.pt` from `Can_Square_Hier_experiment.ipynb` S5), NOT the Lift predictor.

In [ ]:
# Re-define Transformer predictor (same as Module A/B)
class ActionMLP(nn.Module):
    def __init__(self, action_dim=7, hidden_dim=128, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(action_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, a):
        return self.net(a)


class TransformerPredictor(nn.Module):
    def __init__(self, embed_dim=384, action_dim=7, d_model=256,
                 n_layers=4, n_heads=4, max_seq_len=64, dropout=0.2):
        super().__init__()
        self.d_model = d_model
        self.embed_proj = nn.Linear(embed_dim, d_model)
        self.action_mlp = ActionMLP(action_dim, hidden_dim=128, out_dim=d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, batch_first=True,
            dropout=dropout, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, n_layers)
        self.predictor = nn.Linear(d_model, embed_dim)
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, embeds, actions):
        if embeds.dim() == 2:
            embeds = embeds.unsqueeze(1); actions = actions.unsqueeze(1)
        B, T, _ = embeds.shape
        x = self.embed_proj(embeds) + self.action_mlp(actions)
        x = x + self.pos_embed[:, :T, :]
        causal_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        x = self.transformer(x, mask=causal_mask)
        return self.predictor(x[:, -1, :])


# Load task-specific predictor (trained in Can_Square_Hier_experiment.ipynb S5)
pred_ckpt_path = OUTPUT_DIR / f'transformer_{TASK}_K{cfg.rollout_K}_T{cfg.rollout_T}.pt'
assert pred_ckpt_path.exists(), (
    f'Missing {pred_ckpt_path}. Run Can_Square_Hier_experiment.ipynb S5 for TASK={TASK!r} first.'
)
ckpt = torch.load(pred_ckpt_path, map_location=device, weights_only=False)
rollout_predictor = TransformerPredictor(
    embed_dim=cfg.embed_dim, action_dim=cfg.action_dim, d_model=cfg.d_model,
    n_layers=cfg.n_layers_tr, n_heads=cfg.n_heads, dropout=cfg.dropout_p,
).to(device)
rollout_predictor.load_state_dict(ckpt['model_state_dict'])
rollout_predictor.eval()
print(f'Loaded {TASK} predictor K={cfg.rollout_K} T={cfg.rollout_T} (best_cos={ckpt.get("best_cos", "?")})')

In [ ]:
@torch.no_grad()
def generate_synthetic_episode(predictor, z_0, expert_actions, T, sigma, N_perturb, act_noise=0.0):
    """
    Generate N_perturb synthetic trajectories from a perturbed initial state.

    Args:
        predictor: JEPA predictor model (K=1 recommended)
        z_0: initial embedding (384,)
        expert_actions: (ep_len, 7) tensor of expert actions
        T: context window
        sigma: std of Gaussian noise for perturbation
        N_perturb: number of perturbations
    Returns:
        synthetic_z: (N_perturb, ep_len, 384) synthetic embeddings
        synthetic_a: (ep_len, 7) expert actions (repeated)
    """
    ep_len = len(expert_actions)
    synthetic_z = torch.zeros(N_perturb, ep_len, cfg.embed_dim)

    for n in range(N_perturb):
        # Perturb initial embedding
        z_cur = z_0 + torch.randn(cfg.embed_dim) * sigma
        z_cur = z_cur.to(device)
        synthetic_z[n, 0] = z_cur.cpu()

        # Build initial context: repeat z_0 T times
        z_ctx = z_cur.unsqueeze(0).unsqueeze(0).repeat(1, T, 1)
        a_ctx = expert_actions[:1].unsqueeze(0).repeat(1, T, 1).to(device)

        for t in range(1, ep_len):
            # Predict next z
            pred = predictor(z_ctx, a_ctx)  # (1, 384)
            # Slide context
            z_ctx = torch.cat([z_ctx[:, 1:, :], pred.unsqueeze(1)], dim=1)
            a_new = expert_actions[t:t+1].unsqueeze(0).repeat(1, T, 1).to(device)
            a_ctx = a_new
            synthetic_z[n, t] = pred.squeeze(0).cpu()

    return synthetic_z, expert_actions


print('Synthetic generation function ready.')

In [ ]:
# Generate synthetic trajectories for all sigma values (task-specific cache)
for sigma in cfg.sigma_values:
    out_path = EMBED_DIR / f'synthetic_{TASK}_sigma{sigma}.pt'
    if out_path.exists():
        print(f'[CACHED] sigma={sigma} -> {out_path}')
        continue
    print(f'\nGenerating synthetic trajectories with sigma={sigma}...')
    all_synthetic_z = []
    all_synthetic_a = []

    for ep_idx in tqdm(range(cfg.n_train_episodes), desc=f'sigma={sigma}'):
        split_name, start = ep_to_emb_start[ep_idx]
        emb = train_emb if split_name == 'train' else val_emb
        n_frames = all_ep_n_frames[ep_idx]
        if n_frames < 2:
            continue
        z_0 = emb[start]
        expert_actions = all_ep_actions[ep_idx]  # raw actions (not normalized)

        syn_z, syn_a = generate_synthetic_episode(
            rollout_predictor, z_0, expert_actions,
            T=cfg.rollout_T, sigma=sigma, N_perturb=cfg.N_perturb
        )
        all_synthetic_z.append(syn_z)
        all_synthetic_a.append(syn_a)

    torch.save({'z': all_synthetic_z, 'a': all_synthetic_a}, out_path)
    print(f'  Saved {out_path} ({len(all_synthetic_z)} episodes x {cfg.N_perturb} perturb)')

print('\nSynthetic trajectory generation complete.')

In [ ]:
# Quality check: mean cosine similarity between synthetic and real embeddings over rollout steps
fig, ax = plt.subplots(figsize=(8, 5))

for sigma in cfg.sigma_values:
    data = torch.load(EMBED_DIR / f'synthetic_{TASK}_sigma{sigma}.pt', weights_only=False)
    syn_z_list = data['z']  # list of (N_perturb, ep_len, 384)

    all_cos = []
    max_step = 30
    for ep_idx, syn_z in enumerate(syn_z_list[:10]):
        split_name, start = ep_to_emb_start[ep_idx]
        emb = train_emb if split_name == 'train' else val_emb
        real_z = emb[start:start+syn_z.shape[1]]
        steps = min(syn_z.shape[1], real_z.shape[0], max_step)
        cos_per_step = []
        for t in range(steps):
            cs = F.cosine_similarity(syn_z[:, t], real_z[t:t+1].expand(syn_z.shape[0], -1), dim=-1).mean()
            cos_per_step.append(cs.item())
        all_cos.append(cos_per_step)

    min_len = min(len(c) for c in all_cos)
    avg_cos = np.mean([c[:min_len] for c in all_cos], axis=0)
    ax.plot(range(min_len), avg_cos, marker='.', label=f'σ={sigma}')

ax.set_xlabel('Rollout step')
ax.set_ylabel('Mean cos_sim(synthetic, real)')
ax.set_title(f'C2: Synthetic Trajectory Quality ({TASK.upper()})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'c2_synthetic_quality_{TASK}.png', dpi=150, bbox_inches='tight')
plt.show()
print('Quality check done.')

## Section 5: C3 — JEPA-Augmented BC Training

Mix real and synthetic data at ratios $\alpha \in \{0.0, 0.25, 0.5, 0.75, 1.0\}$ and train BC policies.
First select the best $\sigma$ (lowest val MSE at $\alpha=0.5$), then sweep $\alpha$ with that $\sigma$.

In [ ]:
class MixedBCDataset(Dataset):
    """Mix real and synthetic BC samples at ratio alpha.
    alpha=0.0: 100% real; alpha=1.0: 100% synthetic.
    Batch composition: each __getitem__ returns either real or synthetic based on alpha."""
    def __init__(self, real_embeddings, real_actions, ep_n_frames,
                 synthetic_z_list, synthetic_a_list, alpha=0.5):
        self.alpha = alpha
        # Index real samples
        self.real_samples = []
        offset = 0
        for n_frames in ep_n_frames:
            for i in range(n_frames):
                self.real_samples.append((offset + i, offset + i))
            offset += n_frames
        self.real_emb = real_embeddings
        self.real_act = real_actions
        # Flatten synthetic samples
        self.syn_samples = []
        for syn_z, syn_a in zip(synthetic_z_list, synthetic_a_list):
            N_perturb, ep_len, _ = syn_z.shape
            for n in range(N_perturb):
                for t in range(ep_len):
                    self.syn_samples.append((syn_z[n, t], syn_a[t]))
        print(f'Real samples: {len(self.real_samples)}, Synthetic: {len(self.syn_samples)}, α={alpha}')

    def __len__(self):
        return max(len(self.real_samples), len(self.syn_samples))

    def __getitem__(self, idx):
        if torch.rand(1).item() < self.alpha and len(self.syn_samples) > 0:
            si = idx % len(self.syn_samples)
            return self.syn_samples[si]
        else:
            ri = idx % len(self.real_samples)
            emb_i, act_i = self.real_samples[ri]
            return self.real_emb[emb_i], self.real_act[act_i]

print('MixedBCDataset ready.')

In [ ]:
# Quick sigma selection: train BC at alpha=0.5 for each sigma, pick best
sigma_mse = {}

for sigma in cfg.sigma_values:
    print(f'\nTesting σ={sigma} at α=0.5...')
    data = torch.load(EMBED_DIR / f'synthetic_{TASK}_sigma{sigma}.pt', weights_only=False)
    syn_z_list, syn_a_list = data['z'], data['a']

    mixed_ds = MixedBCDataset(train_emb, train_act, train_ep_frames,
                              syn_z_list, syn_a_list, alpha=0.5)
    mixed_ldr = DataLoader(mixed_ds, batch_size=cfg.bc_batch_size, shuffle=True,
                           num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)

    model = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                     hidden_dims=cfg.bc_hidden).to(device)
    result = bc_train_model(model, mixed_ldr, bc_val_ldr, epochs=15, verbose=False)
    sigma_mse[sigma] = result['best_mse']
    print(f'  σ={sigma}: best val MSE = {result["best_mse"]:.6f}')

best_sigma = min(sigma_mse, key=sigma_mse.get)
print(f'\nBest sigma: {best_sigma} (MSE={sigma_mse[best_sigma]:.6f}')

In [ ]:
# Full alpha sweep with best sigma
data = torch.load(EMBED_DIR / f'synthetic_{TASK}_sigma{best_sigma}.pt', weights_only=False)
syn_z_list, syn_a_list = data['z'], data['a']

aug_results = {}

for alpha in cfg.alpha_values:
    print(f'\nTraining BC with α={alpha}...')
    print('-' * 40)

    mixed_ds = MixedBCDataset(train_emb, train_act, train_ep_frames,
                              syn_z_list, syn_a_list, alpha=alpha)
    mixed_ldr = DataLoader(mixed_ds, batch_size=cfg.bc_batch_size, shuffle=True,
                           num_workers=0, pin_memory=(device.type=='cuda'), drop_last=True)

    model = BCPolicy(embed_dim=cfg.embed_dim, action_dim=cfg.action_dim,
                     hidden_dims=cfg.bc_hidden).to(device)
    result = bc_train_model(model, mixed_ldr, bc_val_ldr, epochs=cfg.bc_epochs)
    aug_results[alpha] = result

    # Save checkpoint (task-specific name)
    ckpt = {
        'model_state_dict': model.state_dict(),
        'val_mse': result['best_mse'],
        'alpha': alpha, 'sigma': best_sigma,
        'task': TASK,
    }
    torch.save(ckpt, OUTPUT_DIR / f'module_c_bc_aug_{TASK}_alpha{alpha}.pt')

print('\nAlpha sweep complete.')

In [ ]:
# Plot MSE vs alpha
alphas = sorted(aug_results.keys())
val_mses = [aug_results[a]['best_mse'] for a in alphas]
baseline_mse = bc_result['best_mse']

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(alphas, val_mses, 'o-', linewidth=2, markersize=8, color='steelblue')
ax.axhline(y=baseline_mse, color='red', linestyle='--', linewidth=1.5, label=f'Pure BC (MSE={baseline_mse:.4f})')
ax.set_xlabel('α (synthetic mix ratio)')
ax.set_ylabel('Val MSE')
ax.set_title(f'C3: JEPA-Augmented BC ({TASK.upper()}, σ={best_sigma})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'c3_augmented_bc_{TASK}.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary table
print(f'\n{"α":<8}{"Train MSE":<14}{"Val MSE":<14}{"Δ from baseline":<18}')
print('-' * 54)
for a in alphas:
    r = aug_results[a]
    delta = r['best_mse'] - baseline_mse
    print(f'{a:<8}{r["train_losses"][-1]:<14.6f}{r["best_mse"]:<14.6f}{delta:<+18.6f}')

## Section 6: Save Results

Save all Module C results for this task to a single `.pt` file for use by `module_d_transfer_comparison.ipynb`.

In [ ]:
# Select best alpha (lowest val MSE)
best_alpha = min(aug_results.keys(), key=lambda a: aug_results[a]['best_mse'])
best_aug_mse = aug_results[best_alpha]['best_mse']

results = {
    'task': TASK,
    'bc_baseline_mse': bc_result['best_mse'],
    'best_sigma': best_sigma,
    'sigma_mse': sigma_mse,
    'aug_results': {a: {'best_mse': r['best_mse'], 'train_losses': r['train_losses'], 'val_mses': r['val_mses']}
                       for a, r in aug_results.items()},
    'best_alpha': best_alpha,
    'best_aug_mse': best_aug_mse,
    'baseline_delta': best_aug_mse - bc_result['best_mse'],
}

results_path = OUTPUT_DIR / f'module_c_results_{TASK}.pt'
torch.save(results, results_path)
print(f'Saved {results_path}')

print(f'\n{"="*60}')
print(f'Module C Complete for {TASK.upper()}')
print(f'{"="*60}')
print(f'\nC1 BC baseline: val_mse={bc_result["best_mse"]:.6f}')
print(f'C2 Best sigma: {best_sigma} (MSE={sigma_mse[best_sigma]:.6f})')
print(f'C3 Best alpha: {best_alpha} (val_mse={best_aug_mse:.6f}, Δ={best_aug_mse - bc_result["best_mse"]:+.6f})')

print(f'\nCheckpoints saved:')
for f in sorted(OUTPUT_DIR.glob(f'module_c_*{TASK}*')):
    print(f'  {f.name}')

print(f'\nFigures saved:')
for f in sorted(OUTPUT_DIR.glob(f'c[1-3]_*{TASK}*.png')):
    print(f'  {f.name}')